In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.utils import to_categorical

from tensorflow.keras.callbacks import EarlyStopping

In [2]:
data= pd.read_csv("/content/malicious_phish.csv")
data.head()

,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


In [3]:
sample_size = 15000

benign = data[data['type'] == 'benign'].sample(sample_size, random_state=42)

phishing = data[data['type'] == 'phishing'].sample(sample_size, random_state=42)

defacement = data[data['type'] == 'defacement'].sample(sample_size, random_state=42)

malware = data[data['type'] == 'malware'].sample(sample_size, random_state=42)

data = pd.concat([
    benign,
    phishing,
    defacement,
    malware
])

# Shuffle
data = data.sample(frac=1, random_state=42)

print(data['type'].value_counts())

type
benign        15000
defacement    15000
malware       15000
phishing      15000
Name: count, dtype: int64


In [4]:
extra_benign = [
    # Popular domains
    "yahoo.com",
    "bing.com",
    "duckduckgo.com",
    "mozilla.org",
    "ubuntu.com",
    "python.org",
    "tensorflow.org",
    "keras.io",
    "pytorch.org",
    "oracle.com",
    "ibm.com",
    "salesforce.com",
    "digitalocean.com",
    "medium.com",
    "quora.com",
    "tumblr.com",
    "twitch.tv",
    "steamcommunity.com",
    "epicgames.com",
    "unity.com",
    "unrealengine.com",
    "canva.com",
    "figma.com",
    "behance.net",
    "dribbble.com",
    "airbnb.com",
    "booking.com",
    "uber.com",
    "ola.com",
    "zomato.com",
    "swiggy.com",
    "imdb.com",
    "weather.com",
    "speedtest.net",
    "archive.org",
    "researchgate.net",
    "geeksforgeeks.org",
    "leetcode.com",
    "codeforces.com",
    "hackerrank.com",
    "google.com",
    "github.com",
    "openai.com",
    "microsoft.com",
    "apple.com",
    "stackoverflow.com",
    "kaggle.com",
    "cloudflare.com",
    "adobe.com",
    "intel.com",
    "nvidia.com",
    "facebook.com",
    "instagram.com",
    "twitter.com",
    "x.com",
    "linkedin.com",
    "reddit.com",
    "discord.com",
    "snapchat.com",
    "pinterest.com",
    "youtube.com",
    "netflix.com",
    "spotify.com",
    "primevideo.com",
    "hotstar.com",
    "amazon.com",
    "ebay.com",
    "walmart.com",
    "flipkart.com",
    "wikipedia.org",
    "coursera.org",
    "udemy.com",
    "edx.org",
    "mit.edu",
    "stanford.edu",
    "paypal.com",
    "visa.com",
    "mastercard.com",
    "stripe.com",
    "bbc.com",
    "cnn.com",
    "nytimes.com",
    "reuters.com",
    "notion.so",
    "dropbox.com",
    "zoom.us",
    "slack.com",
    "gitlab.com",
    "vercel.com",
    "huggingface.co",
    "colab.research.google.com"
]

In [5]:
extra_df = pd.DataFrame({
    'url': extra_benign,
    'type': ['benign'] * len(extra_benign)
})

In [6]:
def clean_url(url):

    url = str(url).lower()

    url = url.replace("https://", "")
    url = url.replace("http://", "")
    url = url.replace("www.", "")

    return url

In [7]:
def extract_url_features(url):

    url = str(url).lower()

    features = {}

    # URL length
    features['url_length'] = len(url)

    # Count digits
    features['digit_count'] = sum(c.isdigit() for c in url)

    # Count special chars
    features['special_count'] = len(re.findall(r'[^a-zA-Z0-9]', url))

    # HTTPS
    features['has_https'] = int("https" in url)

    # IP address detection
    features['has_ip'] = int(
        bool(re.search(r'\d+\.\d+\.\d+\.\d+', url))
    )

    # Suspicious keywords
    suspicious_keywords = [
        'login',
        'verify',
        'secure',
        'account',
        'bank',
        'update',
        'free',
        'bonus',
        'crack',
        'paypal',
        'signin'
    ]

    features['suspicious_keywords'] = int(
        any(word in url for word in suspicious_keywords)
    )

    # Dangerous extensions
    features['has_exe'] = int(".exe" in url)
    features['has_php'] = int(".php" in url)

    # Suspicious TLDs
    suspicious_tlds = ['.xyz', '.tk', '.ru', '.biz']

    features['suspicious_tld'] = int(
        any(tld in url for tld in suspicious_tlds)
    )

    return features

In [8]:
data['clean_url'] = data['url'].apply(clean_url)

data[['url', 'clean_url']].head()

,url,clean_url
216883,wmnorthwest.com/issaquah/index.html,wmnorthwest.com/issaquah/index.html
230891,http://www.bissfest-marburg.de/preise/gutschei...,bissfest-marburg.de/preise/gutscheine.pdf
179556,http://www.khd.at/index.php?option=com_rsgalle...,khd.at/index.php?option=com_rsgallery2&gid=14&...
268704,pro-football-reference.com/players/S/StokTi20.htm,pro-football-reference.com/players/s/stokti20.htm
241450,tripadvisor.com/Hotel_Review-g187147-d325719-R...,tripadvisor.com/hotel_review-g187147-d325719-r...


In [9]:
extra_df['clean_url'] = extra_df['url'].apply(clean_url)

In [10]:
data = pd.concat([data, extra_df], ignore_index=True)

data = data.sample(frac=1, random_state=42)

In [11]:
le = LabelEncoder()

data['label'] = le.fit_transform(data['type'])

print(le.classes_)

['benign' 'defacement' 'malware' 'phishing']


In [12]:
print(data['label'].value_counts())

label
0    15091
3    15000
1    15000
2    15000
Name: count, dtype: int64


***TOKENIZATION***

In [13]:
tokenizer = Tokenizer(
    char_level=True
)

tokenizer.fit_on_texts(data['clean_url'])

In [14]:
sequences = tokenizer.texts_to_sequences(data['clean_url'])

print(sequences[0][:20])

[30, 3, 37, 3, 17, 2, 11, 5, 8, 2, 12, 22, 10, 30, 3, 37, 3, 11, 10, 4]


In [15]:
max_length = 200

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print(X.shape)

(60091, 200)


In [16]:
y = to_categorical(data['label'])

print(y.shape)

(60091, 4)


In [17]:
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=data['label']
)

print(x_train.shape)
print(x_test.shape)

(48072, 200)
(12019, 200)


In [18]:
model = Sequential()

# Embedding
model.add(
    Embedding(
        input_dim=len(tokenizer.word_index) + 1,
        output_dim=64,
        input_length=max_length
    )
)

# First BiLSTM
model.add(
    Bidirectional(
        LSTM(64, return_sequences=True)
    )
)

# Second BiLSTM
model.add(
    Bidirectional(
        LSTM(32)
    )
)

# Dropout
model.add(Dropout(0.4))

# Dense
model.add(Dense(32, activation='relu'))

# Output
model.add(Dense(4, activation='softmax'))

# Compile model
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [19]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [20]:
history = model.fit(
    x_train,
    y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/15
301/301 ━━━━━━━━━━━━━━━━━━━━ 21s 43ms/step - accuracy: 0.5641 - loss: 0.9784 - val_accuracy: 0.6542 - val_loss: 0.8210
Epoch 2/15
301/301 ━━━━━━━━━━━━━━━━━━━━ 12s 41ms/step - accuracy: 0.6711 - loss: 0.7934 - val_accuracy: 0.7106 - val_loss: 0.6901
Epoch 3/15
301/301 ━━━━━━━━━━━━━━━━━━━━ 12s 40ms/step - accuracy: 0.7280 - loss: 0.6786 - val_accuracy: 0.7516 - val_loss: 0.6057
Epoch 4/15
301/301 ━━━━━━━━━━━━━━━━━━━━ 12s 40ms/step - accuracy: 0.7766 - loss: 0.5719 - val_accuracy: 0.8023 - val_loss: 0.4939
Epoch 5/15
301/301 ━━━━━━━━━━━━━━━━━━━━ 12s 40ms/step - accuracy: 0.8122 - loss: 0.4931 - val_accuracy: 0.8131 - val_loss: 0.4576
Epoch 6/15
301/301 ━━━━━━━━━━━━━━━━━━━━ 12s 40ms/step - accuracy: 0.8245 - loss: 0.4581 - val_accuracy: 0.8357 - val_loss: 0.4139
Epoch 7/15
301/301 ━━━━━━━━━━━━━━━━━━━━ 21s 41ms/step - accuracy: 0.8424 - loss: 0.4149 - val_accuracy: 0.8435 - val_loss: 0.4040
Epoch 8/15
301/301 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.8474 - loss: 0.3991 - 

In [21]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)

print("Test Accuracy:", test_accuracy)

376/376 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.8665 - loss: 0.3443
Test Accuracy: 0.8665446639060974


In [22]:
y_pred = model.predict(x_test)

y_pred_classes = np.argmax(y_pred, axis=1)

y_true = np.argmax(y_test, axis=1)

376/376 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step


In [23]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred_classes))

              precision    recall  f1-score   support

           0       0.79      0.79      0.79      3019
           1       0.93      0.92      0.93      3000
           2       0.99      0.94      0.97      3000
           3       0.77      0.81      0.79      3000

    accuracy                           0.87     12019
   macro avg       0.87      0.87      0.87     12019
weighted avg       0.87      0.87      0.87     12019



In [24]:
def predict_url_dl(url):

    # Clean URL
    url = clean_url(url)
    features = extract_url_features(url)

    # Convert to sequence
    sequence = tokenizer.texts_to_sequences([url])

    # Pad sequence
    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding='post',
        truncating='post'
    )

    # Predict
    prediction = model.predict(padded, verbose=0)[0]

    predicted_class = np.argmax(prediction)

    confidence = np.max(prediction)

    classes = le.classes_

    predicted_label = classes[predicted_class]

    # Hybrid feature corrections
    # Strong malware indicators
    if features['has_exe'] == 1:
        predicted_label = "malware"

    # Strong defacement indicators
    if (
    features['has_php'] == 1 or
    "1337" in url or
    "hacked" in url
):
        predicted_label = "defacement"

    # Strong phishing indicators
    if (
    features['suspicious_keywords'] == 1 and
    features['suspicious_tld'] == 1 and
    features['has_exe'] == 0
):
        predicted_label = "phishing"

    # Rule-based dangerous URLs bypass threshold
    if predicted_label in ["malware", "defacement"]:
        final_label = predicted_label

    # Normal confidence handling
    elif confidence < 0.85:
        final_label = "benign"

    else:
        final_label = predicted_label

    print("Confidence:", round(confidence * 100, 2), "%")

    return final_label

In [25]:
benign_urls = [
    "https://linkedin.com",
    "https://netflix.com",
    "https://spotify.com",
    "https://leetcode.com",
    "https://huggingface.co",
    "https://drive.google.com",
    "https://chat.openai.com",
    "https://researchgate.net"
]
phishing_urls = [
    "http://paypal-account-verification.xyz",
    "http://bank-login-secure-update.biz",
    "http://amazon-security-alert.tk",
    "http://free-money-bonus.xyz",
    "http://verify-your-account-now.ru"
]
malware_urls = [
    "http://download-crack-software.ru/keygen.exe",
    "http://virus-loader.biz/malware.exe",
    "http://free-hack-tool.tk/trojan.exe",
    "http://darkweb-tools.ru/backdoor.exe"
]
defacement_urls = [
    "http://hacked-site.org/index.php?id=1337",
    "http://anonymous-hacked-page.tk/index.php",
    "http://defaced-portal.com/hacked.php",
    "http://victim-site.net/index.php?admin=hacked"
]
all_tests = [
    ("BENIGN", benign_urls),
    ("PHISHING", phishing_urls),
    ("MALWARE", malware_urls),
    ("DEFACEMENT", defacement_urls)
]

for category, urls in all_tests:

    print("\n" + "="*60)
    print(category)
    print("="*60)

    for url in urls:

        result = predict_url_dl(url)

        print(f"\nURL: {url}")
        print("Prediction:", result)


BENIGN
Confidence: 47.84 %

URL: https://linkedin.com
Prediction: benign
Confidence: 58.6 %

URL: https://netflix.com
Prediction: benign
Confidence: 46.95 %

URL: https://spotify.com
Prediction: benign
Confidence: 55.04 %

URL: https://leetcode.com
Prediction: benign
Confidence: 88.32 %

URL: https://huggingface.co
Prediction: phishing
Confidence: 69.77 %

URL: https://drive.google.com
Prediction: benign
Confidence: 52.92 %

URL: https://chat.openai.com
Prediction: benign
Confidence: 90.11 %

URL: https://researchgate.net
Prediction: phishing

PHISHING
Confidence: 89.21 %

URL: http://paypal-account-verification.xyz
Prediction: phishing
Confidence: 90.09 %

URL: http://bank-login-secure-update.biz
Prediction: phishing
Confidence: 91.99 %

URL: http://amazon-security-alert.tk
Prediction: phishing
Confidence: 90.08 %

URL: http://free-money-bonus.xyz
Prediction: phishing
Confidence: 89.97 %

URL: http://verify-your-account-now.ru
Prediction: phishing

MALWARE
Confidence: 97.05 %

URL: h

In [31]:
# Save BiLSTM model
model.save("bilstm_url_detector.h5")

# Save tokenizer
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save label encoder
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

In [32]:
from google.colab import files

files.download("bilstm_url_detector.h5")
files.download("tokenizer.pkl")
files.download("label_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>